Hola **Juan**!

Soy **Patricio Requena** 👋. Es un placer ser el revisor de tu proyecto el día de hoy!

Revisaré tu proyecto detenidamente con el objetivo de ayudarte a mejorar y perfeccionar tus habilidades. Durante mi revisión, identificaré áreas donde puedas hacer mejoras en tu código, señalando específicamente qué y cómo podrías ajustar para optimizar el rendimiento y la claridad de tu proyecto. Además, es importante para mí destacar los aspectos que has manejado excepcionalmente bien. Reconocer tus fortalezas te ayudará a entender qué técnicas y métodos están funcionando a tu favor y cómo puedes aplicarlos en futuras tareas. 

_**Recuerda que al final de este notebook encontrarás un comentario general de mi parte**_, empecemos!

Encontrarás mis comentarios dentro de cajas verdes, amarillas o rojas, ⚠️ **por favor, no muevas, modifiques o borres mis comentarios** ⚠️:


<div class="alert alert-block alert-success">
<b>Comentario del revisor</b> <a class=“tocSkip”></a>
Si todo está perfecto.
</div>

<div class="alert alert-block alert-warning">
<b>Comentario del revisor</b> <a class=“tocSkip”></a>
Si tu código está bien pero se puede mejorar o hay algún detalle que le hace falta.
</div>

<div class="alert alert-block alert-danger">
<b>Comentario del revisor</b> <a class=“tocSkip”></a>
Si de pronto hace falta algo o existe algún problema con tu código o conclusiones.
</div>

Puedes responderme de esta forma:
<div class="alert alert-block alert-info">
    <b>Respuesta:</b> 

El servicio de venta de autos usados Rusty Bargain está desarrollando una aplicación para atraer nuevos clientes. Gracias a esa app, puedes averiguar rápidamente el valor de mercado de tu coche. Tienes acceso al historial: especificaciones técnicas, versiones de equipamiento y precios. Tienes que crear un modelo que determine el valor de mercado.
A Rusty Bargain le interesa:
- la calidad de la predicción;
- la velocidad de la predicción;
- el tiempo requerido para el entrenamiento

## Preparación de datos

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("/datasets/car_data.csv")
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 354369 entries, 0 to 354368
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   DateCrawled        354369 non-null  object
 1   Price              354369 non-null  int64 
 2   VehicleType        316879 non-null  object
 3   RegistrationYear   354369 non-null  int64 
 4   Gearbox            334536 non-null  object
 5   Power              354369 non-null  int64 
 6   Model              334664 non-null  object
 7   Mileage            354369 non-null  int64 
 8   RegistrationMonth  354369 non-null  int64 
 9   FuelType           321474 non-null  object
 10  Brand              354369 non-null  object
 11  NotRepaired        283215 non-null  object
 12  DateCreated        354369 non-null  object
 13  NumberOfPictures   354369 non-null  int64 
 14  PostalCode         354369 non-null  int64 
 15  LastSeen           354369 non-null  object
dtypes: int64(7), object(

In [3]:
print(df.isna().sum())

DateCrawled              0
Price                    0
VehicleType          37490
RegistrationYear         0
Gearbox              19833
Power                    0
Model                19705
Mileage                  0
RegistrationMonth        0
FuelType             32895
Brand                    0
NotRepaired          71154
DateCreated              0
NumberOfPictures         0
PostalCode               0
LastSeen                 0
dtype: int64


In [4]:
df["VehicleType"] = df["VehicleType"].fillna("Unknown")
df["Gearbox"] = df["Gearbox"].fillna("Unknown")
df["Model"] = df["Model"].fillna("Unknown")
df["FuelType"] = df["FuelType"].fillna("Unknown")
df["NotRepaired"] = df["NotRepaired"].fillna("Unknown")

df['DateCrawled'] = pd.to_datetime(df['DateCrawled'])
df['DateCreated'] = pd.to_datetime(df['DateCreated'])
df['LastSeen'] = pd.to_datetime(df['LastSeen'])

# Features útiles 
df['DaysActive'] = (df['LastSeen'] - df['DateCreated']).dt.days
df['DaysSinceCrawl'] = (df['LastSeen'] - df['DateCrawled']).dt.days

# eliminamos las originales inútiles

df = df.drop(['DateCrawled', 'DateCreated', 'LastSeen','NumberOfPictures'], axis=1)

print(df.isna().sum())

Price                0
VehicleType          0
RegistrationYear     0
Gearbox              0
Power                0
Model                0
Mileage              0
RegistrationMonth    0
FuelType             0
Brand                0
NotRepaired          0
PostalCode           0
DaysActive           0
DaysSinceCrawl       0
dtype: int64


<div class="alert alert-block alert-success">
<b>Comentario del revisor (1ra Iteracion)</b> <a class=“tocSkip”></a>

Muy buen trabajo con el tratamiento y análisis de los datos, siempre en un proyecto lo importante es primero entender los datos con los que se trabajará antes de pasar al modelado
</div>

In [5]:
print(df.duplicated().sum())
df = df.drop_duplicates()

8256


In [6]:
print(df.duplicated().sum())

0


Se rellenaron los valores ausentes del dataframe, se eliminaron valores duplicados y se crearon nuevas features con base en las features de fechas.

## Análisis del modelo

In [7]:
from sklearn.model_selection import train_test_split

X = df.drop("Price", axis= 1)
y = df["Price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=12345)

In [8]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

cat_features = X.select_dtypes(include="object").columns

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
    ],
    remainder="passthrough"
)

model_lr = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

model_lr.fit(X_train, y_train)

y_pred_lr = model_lr.predict(X_test)

In [9]:
from sklearn.tree import DecisionTreeRegressor

model_dt = Pipeline([
    ('prep', preprocessor),
    ('model', DecisionTreeRegressor(max_depth=10, random_state=123))
])

model_dt.fit(X_train, y_train)

y_pred_dt = model_dt.predict(X_test)

In [10]:
from sklearn.ensemble import RandomForestRegressor

model_rf = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=20,
        max_depth=8,
        random_state=12345,
        n_jobs=-1
    ))
])

model_rf.fit(X_train,y_train)

y_pred_rf = model_rf.predict(X_test)

In [11]:
import lightgbm as lgb
from sklearn.preprocessing import OrdinalEncoder

# identificación de columnas categóricas

cat_cols = X.select_dtypes(include="object").columns
num_cols = X.select_dtypes(exclude="object").columns

preprocessor_lgb = ColumnTransformer(
    transformers=[
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols)
    ],
    remainder= "passthrough"
)


model_lgb = Pipeline([
    ("prep", preprocessor_lgb),
    ("lgb", lgb.LGBMRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=10,
    random_state=12345
))
])

model_lgb.fit(X_train,y_train)

y_pred_lgb = model_lgb.predict(X_test)

In [12]:

from sklearn.metrics import mean_squared_error
import numpy as np

def rmse(y_true, y_pred):
    error = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"La raiz del error cuadrático medio del modelo es: {error}")
    return error
   

In [13]:
%%time

model_lr.fit(X_train, y_train)

CPU times: user 640 ms, sys: 2.13 s, total: 2.77 s
Wall time: 1.53 s


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['VehicleType', 'Gearbox', 'Model', 'FuelType', 'Brand', 'NotRepaired'], dtype='object'))])),
                ('model', LinearRegression())])

In [14]:
import time

# Para el modelo de regresión lineal
start = time.time()
model_lr.fit(X_train, y_train)
end = time.time()
print(f"Tiempo de entrenamiento Regresión Lineal: {end - start:.4f} segundos")

Tiempo de entrenamiento Regresión Lineal: 1.4429 segundos


In [ ]:

# Para el modelo de arbol de decisión
start = time.time()
model_dt.fit(X_train, y_train)
end = time.time()
print(f"Tiempo de entrenamiento Arbol de decisión: {end - start:.4f} segundos")

In [ ]:

# Para el modelo de random forest
start = time.time()
model_rf.fit(X_train, y_train)
end = time.time()
print(f"Tiempo de entrenamiento Random Forest: {end - start:.4f} segundos")

<div class="alert alert-block alert-success">
<b>Comentario del revisor (1ra Iteracion)</b> <a class=“tocSkip”></a>

Correcto, se está calculando el tiempo de entrenamiento y predicción por separado lo cual dará información necesaria para cuando se quiera llevar los modelos a un entorno productivo.

Muy buen uso de la función para evitar el duplicado de código para un mismo proceso
</div>

In [ ]:

# Para el modelo de Gradient Boosting
start = time.time()
model_lgb.fit(X_train, y_train)
end = time.time()
print(f"Tiempo de entrenamiento Gradient Boosting: {end - start:.4f} segundos")

In [ ]:
# para calcular el tiempo de predicción

start = time.time()
model_lr.predict(X_test)
end = time.time()
print(f"Tiempo de predicción Regresión Lineal: {end - start:.4f} segundos")


In [ ]:
start = time.time()
model_dt.predict(X_test)
end = time.time()
print(f"Tiempo de predicción arbol de decisión: {end - start:.4f} segundos")


In [ ]:
start = time.time()
model_rf.predict(X_test)
end = time.time()
print(f"Tiempo de predicción Random Forest: {end - start:.4f} segundos")


In [ ]:
start = time.time()
model_lgb.predict(X_test)
end = time.time()
print(f"Tiempo de predicción Gradient Boosting: {end - start:.4f} segundos")


In [ ]:
# Raiz del error cuadrático medio Regresión Lineal:
rmse(y_test, y_pred_lr)

In [ ]:
print(y_test.mean(), y_test.median())

In [ ]:
# Raiz del error cuadrático medio arbol de decisión: 
rmse(y_test, y_pred_dt)

In [ ]:
# Raiz del error cuadrático medio Random Forest:
rmse(y_test, y_pred_rf)

In [ ]:
# Raiz del error cuadrático medio Gradient boosting:
rmse(y_test, y_pred_lgb)

## 📊 Comparación de modelos

| Modelo               | RMSE (€) | Tiempo entrenamiento (s) | Tiempo predicción (s) |
|---------------------|----------|--------------------------|-----------------------|
| Regresión Lineal    | 3360.89  | 1.5510                   | 0.0975                |
| Árbol de Decisión   | 2121.75  | 5.9241                   | 0.0950                |
| Random Forest       | 2144.59  | 24.3259                  | 0.1306                |
| Gradient Boosting (LightGBM) | 1764.51  | 4.6059                   | 0.4440                |

## 🧠 Conclusión

En este proyecto se compararon diferentes modelos de regresión para predecir el precio de vehículos usados, evaluando tanto la calidad de las predicciones como la eficiencia computacional.

La regresión lineal se utilizó como modelo base (baseline), obteniendo un RMSE de 3360.89 €, lo cual indica un rendimiento limitado debido a la incapacidad del modelo para capturar relaciones no lineales en los datos.

Los modelos basados en árboles mostraron una mejora significativa. El árbol de decisión redujo el RMSE a 2121.75 €, mientras que Random Forest obtuvo un rendimiento similar (2144.59 €), aunque con un costo computacional mucho mayor en tiempo de entrenamiento.

El mejor desempeño lo obtuvo el modelo de Gradient Boosting (LightGBM), con un RMSE de 1764.51 €, lo que representa una mejora considerable respecto a los demás modelos. Además, su tiempo de entrenamiento fue relativamente bajo en comparación con Random Forest, aunque su tiempo de predicción fue el más alto.

Estos resultados indican que LightGBM ofrece el mejor balance entre precisión y eficiencia para este problema. Su capacidad para modelar relaciones complejas y manejar variables categóricas lo convierte en la opción más adecuada para estimar el valor de mercado de vehículos en este contexto.

Finalmente, se observa que la distribución del precio presenta sesgo hacia valores altos (outliers), lo cual impacta la métrica RMSE. Aun así, los modelos de boosting logran manejar mejor esta complejidad en comparación con los modelos más simples.

<div class="alert alert-block alert-success">
<b>Comentario general (1ra Iteracion)</b> <a class=“tocSkip”></a>

Muy bien, se evaluaron correctamente las métricas solicitadas, en cuanto a las métricas de desempeño de las predicciones cómo en el tiempo que toma cada una de sus etapas. El medir el tiempo es importante ya que cuando pones los modelos en producción para que puedan ser usados se suele realizar por medio de API donde se prioriza más el tiempo de predicción que la precisión cómo tal. Lo ideal es buscar siempre el balance entre buen desempeño de predicciones y de tiempo de predicción para tu modelo, sobre todo cuando va a ser usado en tiempo real.
    
Saludos!
</div>

# Lista de control

Escribe 'x' para verificar. Luego presiona Shift+Enter

- [x]  Jupyter Notebook está abierto
- [ ]  El código no tiene errores- [ ]  Las celdas con el código han sido colocadas en orden de ejecución- [ ]  Los datos han sido descargados y preparados- [ ]  Los modelos han sido entrenados
- [ ]  Se realizó el análisis de velocidad y calidad de los modelos